<a href="https://colab.research.google.com/github/kawastony/grok-notes-version-2/blob/main/Lattice_L_values.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q primme
import primme
print('primme OK', getattr(primme, '__version__', 'unknown'))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.8/626.8 kB 5.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import numpy as np
from itertools import product
from scipy.sparse import coo_matrix, diags
from scipy.sparse.linalg import eigsh, LinearOperator
import time, gc, warnings
warnings.filterwarnings('ignore')

I2 = np.eye(2, dtype=complex)
sx = np.array([[0., 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
z2 = np.zeros((2, 2), dtype=complex)
alpha4 = [np.block([[z2, s], [s, z2]]) for s in (sx, sy, sz)]
beta4 = np.block([[I2, z2], [z2, -I2]])
ALPHA = [np.kron(a, I2).astype(complex) for a in alpha4]
BETA = np.kron(beta4, I2).astype(complex)
G5 = np.kron(beta4, I2).astype(complex)
MASS_B = [np.kron(beta4, t).astype(complex) for t in (sx, sy, sz)]
EYE8 = np.eye(8, dtype=complex)
R = 1.0
G5DIAG = np.diag(G5).real.copy()
print('Algebra OK')

In [ ]:
def hedgehog(x, y, z, cx, cy, cz, L, w):
    rx = (x - cx) - L * int(round((x - cx) / float(L)))
    ry = (y - cy) - L * int(round((y - cy) / float(L)))
    rz = (z - cz) - L * int(round((z - cz) / float(L)))
    rr = np.sqrt(rx*rx + ry*ry + rz*rz + 1e-16)
    f = np.tanh(rr / w)
    th = np.arccos(np.clip(rz / rr, -1., 1.))
    ph = np.arctan2(ry, rx)
    hx = np.sin(th)*np.cos(ph); hy = np.sin(th)*np.sin(ph); hz = np.cos(th)
    d = rr / np.sqrt(rr*rr + 0.08)
    hx, hy, hz = d*hx, d*hy, d*hz
    n = np.sqrt(hx*hx + hy*hy + hz*hz + 1e-30)
    return f, hx/n, hy/n, hz/n

def precompute_onsite(L, m0, v, w, sep):
    cx1, cy1, cz1 = L//2, L//2, (L//2 - sep//2) % L
    cx2, cy2, cz2 = L//2, L//2, (L//2 + sep//2) % L
    M = np.zeros((L, L, L, 8, 8), dtype=complex)
    base = m0*BETA + 3*R*EYE8
    for x, y, z in product(range(L), repeat=3):
        Md = base.copy()
        for sign, cx, cy, cz in [(+1, cx1, cy1, cz1), (-1, cx2, cy2, cz2)]:
            f, hx, hy, hz = hedgehog(x, y, z, cx, cy, cz, L, w)
            if f < 1e-14:
                continue
            Md = Md + sign*v*f*(hx*MASS_B[0] + hy*MASS_B[1] + hz*MASS_B[2])
        M[x, y, z] = Md
    return M, (cx1, cy1, cz1), (cx2, cy2, cz2)

def apply_H(psi_flat, L, M_on):
    psi = psi_flat.reshape((L, L, L, 8))
    out = np.einsum('xyzab,xyzb->xyza', M_on, psi)
    for mu, axis in enumerate([0, 1, 2]):
        Aplus = -0.5*(R*EYE8 - ALPHA[mu])
        Aminus = -0.5*(R*EYE8 + ALPHA[mu])
        out = out + np.einsum('ab,xyzb->xyza', Aplus, np.roll(psi, -1, axis=axis))
        out = out + np.einsum('ab,xyzb->xyza', Aminus, np.roll(psi, +1, axis=axis))
    out = np.einsum('ab,xyzb->xyza', G5, out)
    return out.reshape(-1)

def build_sparse_H(L, m0, v, w, sep):
    Nspin = 8
    N = Nspin * L**3
    cx1, cy1, cz1 = L//2, L//2, (L//2 - sep//2) % L
    cx2, cy2, cz2 = L//2, L//2, (L//2 + sep//2) % L
    rows, cols, data = [], [], []
    def add(i0, j0, mat):
        for s in range(8):
            for sp in range(8):
                val = mat[s, sp]
                if abs(val) > 1e-18:
                    rows.append(i0 + s)
                    cols.append(j0 + sp)
                    data.append(complex(val))
    base = m0*BETA + 3*R*EYE8
    for x, y, z in product(range(L), repeat=3):
        i0 = Nspin * (z + L*(y + L*x))
        Md = base.copy()
        for sign, cx, cy, cz in [(+1, cx1, cy1, cz1), (-1, cx2, cy2, cz2)]:
            f, hx, hy, hz = hedgehog(x, y, z, cx, cy, cz, L, w)
            if f < 1e-14:
                continue
            Md = Md + sign*v*f*(hx*MASS_B[0] + hy*MASS_B[1] + hz*MASS_B[2])
        add(i0, i0, Md)
        for mu, (dx, dy, dz) in enumerate([(1,0,0), (0,1,0), (0,0,1)]):
            jp = Nspin * (((z+dz)%L) + L*(((y+dy)%L) + L*((x+dx)%L)))
            jm = Nspin * (((z-dz)%L) + L*(((y-dy)%L) + L*((x-dx)%L)))
            add(i0, jp, -0.5*(R*EYE8 - ALPHA[mu]))
            add(i0, jm, -0.5*(R*EYE8 + ALPHA[mu]))
    D = coo_matrix((data, (rows, cols)), shape=(N, N), dtype=complex).tocsr()
    G = diags(np.tile(G5DIAG, L**3))
    GD = G @ D
    return 0.5*(GD + GD.getH()).tocsr()

print('Operators defined')

In [ ]:
L, m0, v, w, sep = 8, 0.3, 2.0, 1.0, 5
print(f'Reference L={L} ...')
t0 = time.time()
Hs = build_sparse_H(L, m0, v, w, sep)
ev_ref, evec_ref = eigsh(Hs, k=4, sigma=0.0, which='LM', maxiter=4000, tol=1e-6)
ord = np.argsort(np.abs(ev_ref))
ev_ref, evec_ref = ev_ref[ord], evec_ref[:, ord]
print(f'sparse |λ| = {np.round(np.abs(ev_ref), 6)}  ({time.time()-t0:.1f}s)')

In [ ]:
print('PRIMME sparse L=8 ...')
t0 = time.time()
try:
    ev_p, evec_p = primme.eigsh(Hs, k=4, which='SM', tol=1e-6, maxiter=10000)
    ord = np.argsort(np.abs(ev_p))
    ev_p, evec_p = ev_p[ord], evec_p[:, ord]
    print(f'PRIMME |λ| = {np.round(np.abs(ev_p), 6)}  ({time.time()-t0:.1f}s)')
    print(f'max |λ| diff vs ARPACK = {np.max(np.abs(np.abs(ev_p) - np.abs(ev_ref))):.2e}')
    ovs = [float(np.max(np.abs(evec_ref[:, i].conj() @ evec_p))) for i in range(4)]
    print(f'subspace overlaps = {np.round(ovs, 4)}')
except Exception as e:
    print(f'PRIMME sparse failed: {type(e).__name__}: {e}')

In [ ]:
M_on, c1, c2 = precompute_onsite(L, m0, v, w, sep)
N = 8 * L**3

def matvec(x):
    return apply_H(x, L, M_on)

Hop = LinearOperator((N, N), matvec=matvec, dtype=np.complex128)

print('PRIMME matrix-free L=8 ...')
t0 = time.time()
try:
    ev_mf, evec_mf = primme.eigsh(Hop, k=4, which='SM', tol=1e-5, maxiter=20000)
    ord = np.argsort(np.abs(ev_mf))
    ev_mf, evec_mf = ev_mf[ord], evec_mf[:, ord]
    print(f'PRIMME MF |λ| = {np.round(np.abs(ev_mf), 6)}  ({time.time()-t0:.1f}s)')
    print(f'max |λ| diff vs ARPACK = {np.max(np.abs(np.abs(ev_mf) - np.abs(ev_ref))):.2e}')
except Exception as e:
    print(f'PRIMME MF failed: {type(e).__name__}: {e}')
    print('Try which="SA" on H^2 path or increase maxiter')

In [ ]:
L, sep = 10, 6
print(f'Reference L={L} ...')
Hs10 = build_sparse_H(L, m0, v, w, sep)
ev10, _ = eigsh(Hs10, k=4, sigma=0.0, which='LM', maxiter=5000, tol=1e-5)
ev10 = np.sort(np.abs(ev10))
print(f'sparse |λ| = {np.round(ev10, 6)}')

print('PRIMME sparse L=10 ...')
t0 = time.time()
try:
    ev_p10, _ = primme.eigsh(Hs10, k=4, which='SM', tol=1e-5, maxiter=15000)
    ev_p10 = np.sort(np.abs(ev_p10))
    print(f'PRIMME |λ| = {np.round(ev_p10, 6)}  ({time.time()-t0:.1f}s)')
    print(f'diff = {np.max(np.abs(ev_p10 - ev10)):.2e}')
except Exception as e:
    print(f'failed: {e}')

In [ ]:
L, sep = 12, 6
print(f'L={L} matrix-free PRIMME (N={8*L**3}) ...')
M_on12, _, _ = precompute_onsite(L, m0, v, w, sep)
N12 = 8 * L**3

def matvec12(x):
    return apply_H(x, L, M_on12)

Hop12 = LinearOperator((N12, N12), matvec=matvec12, dtype=np.complex128)

t0 = time.time()
try:
    ev12, evec12 = primme.eigsh(
        Hop12, k=4, which='SM',
        tol=1e-4, maxiter=30000,
        # method='PRIMME_JDQMR',  # optional
    )
    ord = np.argsort(np.abs(ev12))
    ev12 = ev12[ord]
    print(f'L=12 PRIMME |λ| = {np.round(np.abs(ev12), 6)}  ({time.time()-t0:.1f}s)')
    print('L=12 SUCCESS — soft modes extracted matrix-free')
except Exception as e:
    print(f'L=12 failed: {type(e).__name__}: {e}')
    print(f'elapsed {time.time()-t0:.1f}s')

In [ ]:
# Run only if you have soft eigenvectors evec_s and L, M_on, sep
def chi_mass_texture(psi, L, w, sep):
    cx1, cy1, cz1 = L//2, L//2, (L//2 - sep//2) % L
    cx2, cy2, cz2 = L//2, L//2, (L//2 + sep//2) % L
    pr = psi.reshape((L, L, L, 8))
    acc = 0.0
    for x, y, z in product(range(L), repeat=3):
        vx = vy = vz = 0.0
        for sign, cx, cy, cz in [(+1, cx1, cy1, cz1), (-1, cx2, cy2, cz2)]:
            f, hx, hy, hz = hedgehog(x, y, z, cx, cy, cz, L, w)
            vx += sign*f*hx; vy += sign*f*hy; vz += sign*f*hz
        n = np.sqrt(vx*vx + vy*vy + vz*vz + 1e-30)
        M = (vx/n)*MASS_B[0] + (vy/n)*MASS_B[1] + (vz/n)*MASS_B[2]
        acc += np.vdot(pr[x, y, z], M @ pr[x, y, z]).real
    return float(acc)

# Example if evec_mf exists from Cell 6:
# for i in range(4):
#     print(i, chi_mass_texture(evec_mf[:, i], 8, w, 5))

In [ ]:
ev_p, evec_p = primme.eigsh(Hs, k=4, which='SM', tol=1e-6, maxiter=50000)

In [ ]:
L, sep = 10, 6
M_on10, _, _ = precompute_onsite(L, m0, v, w, sep)
N10 = 8 * L**3
Hop10 = LinearOperator((N10, N10), matvec=lambda x: apply_H(x, L, M_on10), dtype=np.complex128)
t0 = time.time()
ev10p, evec10p = primme.eigsh(Hop10, k=4, which='SM', tol=1e-5, maxiter=50000)
print(np.round(np.abs(np.sort_complex(ev10p)), 6), f'{time.time()-t0:.1f}s')

In [ ]:
L, sep = 12, 6
M_on12, _, _ = precompute_onsite(L, m0, v, w, sep)
N12 = 8 * L**3
Hop12 = LinearOperator((N12, N12), matvec=lambda x: apply_H(x, L, M_on12), dtype=np.complex128)
t0 = time.time()
try:
    ev12, evec12 = primme.eigsh(
        Hop12, k=4, which='SM',
        tol=1e-4, maxiter=100000,
        method='PRIMME_JDQMR',
    )
    print('L=12', np.round(np.abs(np.sort_complex(ev12)), 6), f'{time.time()-t0:.1f}s')
except Exception as e:
    print(type(e).__name__, e, f'elapsed {time.time()-t0:.1f}s')

In [ ]:
# If C fails, try very loose tol to see if any soft pair appears
try:
    ev12b, evec12b = primme.eigsh(Hop12, k=4, which='SM', tol=1e-3, maxiter=80000)
    print(np.round(np.abs(np.sort_complex(ev12b)), 5))
except Exception as e:
    print(f"PRIMME eigsh failed in slbU1Fe3XDGQ: {type(e).__name__}: {e}")

In [ ]:
# After the loose primme.eigsh that returned ev12b, evec12b:
ev = np.asarray(ev12b).real
vec = np.asarray(evec12b)
ord = np.argsort(np.abs(ev))
ev, vec = ev[ord], vec[:, ord]

print('candidates |λ|:', np.round(np.abs(ev), 6))
for i in range(min(4, vec.shape[1])):
    r = apply_H(vec[:, i], 12, M_on12) - ev[i] * vec[:, i]
    print(i, 'res', np.linalg.norm(r), 'rel', np.linalg.norm(r)/(np.linalg.norm(ev[i]*vec[:, i])+1e-30))

In [ ]:
t0 = time.time()
try:
    ev12, evec12 = primme.eigsh(
        Hop12, k=4, which='SM',
        tol=5e-4, maxiter=200000,
        method='PRIMME_JDQMR',
    )
    print(np.round(np.abs(np.sort_complex(ev12)), 6), f'{time.time()-t0:.1f}s')
except Exception as e:
    print(type(e).__name__, e, f'{time.time()-t0:.1f}s')

In [ ]:
print(f'L={L} matrix-free PRIMME (N={N12}) - Tighter run (increased maxiter) ...')
t0 = time.time()
try:
    ev12, evec12 = primme.eigsh(
        Hop12, k=4, which='SM',
        tol=5e-4, maxiter=1000000, # Increased maxiter again
        method='PRIMME_JDQMR',
    )
    ord = np.argsort(np.abs(ev12))
    ev12_converged, evec12_converged = ev12[ord], evec12[:, ord]
    print(f'L=12 PRIMME |λ| = {np.round(np.abs(ev12_converged), 6)}  ({time.time()-t0:.1f}s)')
    print('L=12 SUCCESS - Soft modes extracted matrix-free with tighter parameters')

    # Calculate and print residuals for the converged modes
    print('\nVerifying residuals:')
    for i in range(min(4, evec12_converged.shape[1])):
        r = apply_H(evec12_converged[:, i], L, M_on12) - ev12_converged[i] * evec12_converged[:, i]
        res_norm = np.linalg.norm(r)
        # Avoid division by zero if eigenvalue is extremely small
        rel_res = res_norm / (np.linalg.norm(ev12_converged[i] * evec12_converged[:, i]) + 1e-30)
        print(f'{i} res {res_norm:.2e}, rel {rel_res:.2e}')

except Exception as e:
    print(f'L=12 tighter run failed: {type(e).__name__}: {e}')
    print(f'elapsed {time.time()-t0:.1f}s')
    ev12_converged = None # Indicate no convergence
    evec12_converged = None

If the previous cell succeeded and the relative residuals are `≲ 10⁻³`, we can now confidently compute the `chi_mass_texture` for the L=12 soft modes.

In [ ]:
# Run this cell only if 'ev12_converged' and 'evec12_converged' exist from the previous cell
if ev12_converged is not None and evec12_converged is not None:
    print('\nComputing chi_mass_texture for converged L=12 modes:')
    for i in range(min(4, evec12_converged.shape[1])):
        # Ensure 'w' and 'sep' are correctly defined from earlier cells, currently they are global
        print(f'{i} chi: {chi_mass_texture(evec12_converged[:, i], L, w=w, sep=sep):.4f}')
else:
    print('Skipping chi_mass_texture calculation as L=12 modes did not converge in the previous step.')

In [ ]:
# only if relative residuals ≲ 1e-2 or better
for i in range(4):
    print(i, chi_mass_texture(vec[:, i], 12, w=1.0, sep=6))

# L=12 Soft Mode Analysis: Texture-Coherent Molecularity and Bridge Propagation

This section implements the detailed test plan to investigate the properties of the lowest soft modes at $L=12$, focusing on their alignment with mass texture and potential bridge formation between cores. We will scan different separations and compute key observables for interpretation.

## 1. Setup and Loading Eigenmodes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.sparse.linalg import LinearOperator
from itertools import product # Added missing import
import time # Added missing import

# Define L, m0, v, w globally for this analysis
L = 12
m0 = 0.3
v = 2.0
w = 1.0

# Define parameters for the scan
d_list = [1, 2, 3, 4, 5, 6] # Separations to test
k_modes = 4                # Number of eigenmodes to extract
rcore = 1.5                # Core radius for masks
r_bridge = 1               # Bridge radius for masks

# Initialize a list to store all computed observables
all_observables = []

# Define core coordinates for L=12 (assuming L is an even number here)
cx_global, cy_global = L // 2, L // 2

# Pre-calculate (x,y,z) coordinates for convenience in masking and plotting
coords = np.array(list(product(range(L), repeat=3)))
x_coords = coords[:, 0]
y_coords = coords[:, 1]
z_coords = coords[:, 2]

print(f"Starting L={L} analysis with separations: {d_list}")

## 2. Helper Functions for Geometry Masks and Local Observables

These functions help define the spatial regions (cores, bridge) and compute per-site quantities like density and texture alignment.

In [ ]:
def make_core_masks(L, current_sep, r_core):
    """Creates boolean masks for core regions C1 and C2."""
    c1_z = (L // 2 - current_sep // 2) % L
    c2_z = (L // 2 + current_sep // 2) % L

    C1 = np.zeros((L, L, L), dtype=bool)
    C2 = np.zeros((L, L, L), dtype=bool)

    for x, y, z in product(range(L), repeat=3):
        # Periodic boundary conditions for distance calculation
        rx1 = (x - cx_global) - L * round((x - cx_global) / L)
        ry1 = (y - cy_global) - L * round((y - cy_global) / L)
        rz1 = (z - c1_z) - L * round((z - c1_z) / L)
        dist1_sq = rx1**2 + ry1**2 + rz1**2

        rx2 = (x - cx_global) - L * round((x - cx_global) / L)
        ry2 = (y - cy_global) - L * round((y - cy_global) / L)
        rz2 = (z - c2_z) - L * round((z - c2_z) / L)
        dist2_sq = rx2**2 + ry2**2 + rz2**2

        if dist1_sq <= r_core**2:
            C1[x, y, z] = True
        if dist2_sq <= r_core**2:
            C2[x, y, z] = True

    return C1, C2

def make_bridge_mask(L, current_sep, r_bridge):
    """Creates a boolean mask for the bridge region B."""
    # Assuming bridge is along the z-axis between cores
    c1_z = (L // 2 - current_sep // 2) % L
    c2_z = (L // 2 + current_sep // 2) % L

    # Ensure c1_z is always smaller for consistent range (handling wrap-around)
    if c1_z > c2_z:
        c1_z, c2_z = c2_z, c1_z

    B = np.zeros((L, L, L), dtype=bool)

    for x, y, z in product(range(L), repeat=3):
        # Transverse distance from center (cx_global, cy_global)
        rx_t = (x - cx_global) - L * round((x - cx_global) / L)
        ry_t = (y - cy_global) - L * round((y - cy_global) / L)
        r_perp_sq = rx_t**2 + ry_t**2

        is_in_longitudinal_range = False
        if c1_z < c2_z:
            # Normal case: c1_z --- B --- c2_z
            if z > c1_z and z < c2_z:
                is_in_longitudinal_range = True
        else:
            # Wrapped around case: B -- c2_z --- c1_z -- B (this case is complex)
            # For simplicity, if c1_z > c2_z after ensuring c1_z < c2_z, it means the cores are on opposite sides of the periodic boundary.
            # We'll treat the bridge as the region NOT occupied by the cores directly along the axis.
            # This definition of 'bridge' for wrap-around may need refinement.
            # For now, let's assume `sep` is small enough that cores are not wrapping around in a complex way.
            # A simpler definition for the bridge might be all z-slices between c1_z and c2_z.
            # Let's define it as all z values that are not the exact z-coordinates of the cores.
            if z != c1_z and z != c2_z:
                is_in_longitudinal_range = True

        if r_perp_sq <= r_bridge**2 and is_in_longitudinal_range:
            B[x, y, z] = True
    return B

def get_local_mass_texture_operator_field(L, w, current_sep):
    """Precomputes the local mass texture operator M for each site."""
    M_field = np.empty((L, L, L, 8, 8), dtype=complex)

    c1_z = (L // 2 - current_sep // 2) % L
    c2_z = (L // 2 + current_sep // 2) % L

    for x, y, z in product(range(L), repeat=3):
        vx = vy = vz = 0.0
        for sign, cx, cy, cz in [(+1, cx_global, cy_global, c1_z), (-1, cx_global, cy_global, c2_z)]:
            f, hx, hy, hz = hedgehog(x, y, z, cx, cy, cz, L, w)
            vx += sign * f * hx
            vy += sign * f * hy
            vz += sign * f * hz
        n = np.sqrt(vx * vx + vy * vy + vz * vz + 1e-30)
        if n > 1e-18: # Avoid division by zero
            M_field[x, y, z] = (vx / n) * MASS_B[0] + (vy / n) * MASS_B[1] + (vz / n) * MASS_B[2]
        else:
            M_field[x, y, z] = np.zeros((8, 8), dtype=complex)

    return M_field

def local_density_field(psi_n, L):
    """Computes local density rho_n(x) for a mode psi_n."""
    pr = psi_n.reshape((L, L, L, 8))
    rho = np.sum(np.abs(pr)**2, axis=-1)
    return rho

def calculate_chi_T_field(psi_n, L, M_field_local):
    """Computes local texture alignment chi_T_n(x) for a mode psi_n."""
    pr = psi_n.reshape((L, L, L, 8))
    chi_T_field = np.zeros((L, L, L), dtype=float)
    for x, y, z in product(range(L), repeat=3):
        # The per-site contribution from chi_mass_texture
        chi_T_field[x, y, z] = np.vdot(pr[x, y, z], M_field_local[x, y, z] @ pr[x, y, z]).real
    return chi_T_field

## 3. Iterating Through Separations and Computing Observables

In [ ]:
for current_sep in d_list:
    print(f'\nProcessing L={L}, separation d={current_sep}...')

    # Rebuild Hamiltonian for current separation
    M_on_sep, _, _ = precompute_onsite(L, m0, v, w, current_sep)
    N_total = 8 * L**3

    def matvec_sep(x):
        return apply_H(x, L, M_on_sep)

    Hop_sep = LinearOperator((N_total, N_total), matvec=matvec_sep, dtype=np.complex128)

    t0 = time.time()
    try:
        # Using parameters that showed some success, or increased maxiter
        eigvals_raw, eigvecs_raw = primme.eigsh(
            Hop_sep, k=k_modes, which='SM',
            tol=5e-4, maxiter=150000, # Increased maxiter for better convergence
            method='PRIMME_JDQMR',
        )

        # Sort eigenvalues by absolute value
        ord_idx = np.argsort(np.abs(eigvals_raw))
        eigvals, eigvecs = eigvals_raw[ord_idx], eigvecs_raw[:, ord_idx]

        print(f'  PRIMME MF |λ| = {np.round(np.abs(eigvals), 6)} ({time.time()-t0:.1f}s)')
        print(f'  Successfully extracted {k_modes} modes.')

        # Precompute local mass texture operator field once per separation
        M_field_local = get_local_mass_texture_operator_field(L, w, current_sep)

        # Define masks for current separation
        C1_mask, C2_mask = make_core_masks(L, current_sep, r_core=rcore)
        B_mask = make_bridge_mask(L, current_sep, r_bridge=r_bridge)

        for n in range(k_modes):
            psi_n = eigvecs[:, n] # eigenvector for mode n

            # 1. Density
            rho_n = local_density_field(psi_n, L) # (L,L,L) array

            # 2. Texture-alignment field
            chiT_n = calculate_chi_T_field(psi_n, L, M_field_local) # (L,L,L) array

            # Positive anti-alignment weight
            wT_n = rho_n * np.maximum(0.0, -chiT_n)

            # 3. Core support
            P1_n = np.sum(rho_n[C1_mask])
            P2_n = np.sum(rho_n[C2_mask])
            Prest_n = 1.0 - P1_n - P2_n # Assuming psi_n is normalized

            # 4. Core texture signature
            X1_n = np.sum(chiT_n[C1_mask])
            X2_n = np.sum(chiT_n[C2_mask])

            epsilon = 1e-12 # Small value to prevent division by zero
            t1_n = X1_n / (P1_n + epsilon)
            t2_n = X2_n / (P2_n + epsilon)

            ST_n = np.sign(t1_n) * np.sign(t2_n)

            # 5. Global texture charge
            QT_n = np.sum(chiT_n)

            # 6. Bridge observables
            Brho_n = np.sum(rho_n[B_mask])
            BT_n = np.sum(wT_n[B_mask])

            # 7. Texture-weighted anisotropy
            # Reshape coordinates to (L,L,L) for element-wise multiplication with wT_n
            z_coords_reshaped = z_coords.reshape((L, L, L))
            x_coords_reshaped = x_coords.reshape((L, L, L))
            y_coords_reshaped = y_coords.reshape((L, L, L))

            z2_wT_sum = np.sum(z_coords_reshaped**2 * wT_n)
            r2_wT_sum = np.sum((x_coords_reshaped**2 + y_coords_reshaped**2) * wT_n)
            AT_n = z2_wT_sum / (r2_wT_sum + epsilon)

            # Store all observables for this mode and separation
            all_observables.append({
                "L": L, "d": current_sep, "n": n, "lambda": np.abs(eigvals[n]),
                "P1": P1_n, "P2": P2_n, "Prest": Prest_n,
                "QT": QT_n, "X1": X1_n, "X2": X2_n, "t1": t1_n, "t2": t2_n, "ST": ST_n,
                "Brho": Brho_n, "BT": BT_n, "AT": AT_n
            })

    except Exception as e:
        print(f'  PRIMME MF failed for d={current_sep}: {type(e).__name__}: {e}')
        print(f'  Elapsed {time.time()-t0:.1f}s. Skipping this separation.')

# Convert results to DataFrame
results_df = pd.DataFrame(all_observables)
display(results_df.head())
print(f"\nTotal {len(results_df)} observable sets collected.")

## 4. Diagnostic Plots

Visualize the core observables across different separations $d$ for modes $n=0, 1$ (the lowest two soft modes).

In [ ]:
if not results_df.empty:
    fig, axes = plt.subplots(4, 1, figsize=(10, 20), sharex=True)
    fig.suptitle(f'L={L} Soft Mode Observables vs. Separation (d)', fontsize=16)

    # Filter for modes 0 and 1 if available
    modes_to_plot = results_df[results_df['n'].isin([0, 1])]

    if not modes_to_plot.empty:
        # 1. QT_n vs d
        sns.lineplot(ax=axes[0], data=modes_to_plot, x='d', y='QT', hue='n', marker='o', palette='viridis')
        axes[0].set_title('Global Texture Charge (QT)')
        axes[0].set_ylabel('$Q_{T,n}$', rotation=0, ha='right')
        axes[0].legend(title='Mode')
        axes[0].grid(True, linestyle='--', alpha=0.7)

        # 2. t1_n, t2_n vs d
        sns.lineplot(ax=axes[1], data=modes_to_plot, x='d', y='t1', hue='n', marker='x', linestyle='--', label='t1', palette='viridis')
        sns.lineplot(ax=axes[1], data=modes_to_plot, x='d', y='t2', hue='n', marker='o', linestyle='-', label='t2', palette='viridis', legend=False)
        axes[1].set_title('Normalized Core Texture Signatures (t1, t2)')
        axes[1].set_ylabel('$\bar t_{k,n}$', rotation=0, ha='right')
        # Manually create legend for t1/t2 and modes
        handles, labels = axes[1].get_legend_handles_labels()
        # Filter out duplicate labels for modes from the second lineplot
        unique_labels = {} # Using a dict to preserve order for mode labels
        for h, l in zip(handles, labels):
            unique_labels[l] = h
        # Sort unique_labels by label for consistent legend order
        sorted_unique_labels = sorted(unique_labels.items())
        # Recreate handles and labels with 't1', 't2' and then mode '0', '1'
        legend_handles = [h for l, h in sorted_unique_labels if l.isdigit()] # Get mode handles first
        legend_labels = [f'mode {l}' for l, h in sorted_unique_labels if l.isdigit()] # Get mode labels first

        # Add 't1' and 't2' specific labels and corresponding line styles/markers to legend
        if 't1' in unique_labels: legend_handles.append(plt.Line2D([], [], color='gray', linestyle='--', marker='x')); legend_labels.append('$\bar t_{1,n}$ (Core 1)')
        if 't2' in unique_labels: legend_handles.append(plt.Line2D([], [], color='gray', linestyle='-', marker='o')); legend_labels.append('$\bar t_{2,n}$ (Core 2)')
        axes[1].legend(handles=legend_handles, labels=legend_labels, title='Observable/Mode')
        axes[1].grid(True, linestyle='--', alpha=0.7)

        # 3. BT_n vs d
        sns.lineplot(ax=axes[2], data=modes_to_plot, x='d', y='BT', hue='n', marker='o', palette='viridis')
        axes[2].set_title('Texture-Weighted Bridge Density (BT)')
        axes[2].set_ylabel('$B_{T,n}$', rotation=0, ha='right')
        axes[2].legend(title='Mode')
        axes[2].grid(True, linestyle='--', alpha=0.7)

        # 4. P1_n, P2_n, Prest_n vs d
        sns.lineplot(ax=axes[3], data=modes_to_plot, x='d', y='P1', hue='n', marker='x', linestyle='--', label='P1', palette='viridis')
        sns.lineplot(ax=axes[3], data=modes_to_plot, x='d', y='P2', hue='n', marker='o', linestyle='-', label='P2', palette='viridis', legend=False)
        sns.lineplot(ax=axes[3], data=modes_to_plot, x='d', y='Prest', hue='n', marker='^', linestyle=':', label='Prest', palette='viridis', legend=False)
        axes[3].set_title('Core Support and Rest (P1, P2, Prest)')
        axes[3].set_xlabel('Separation d')
        axes[3].set_ylabel('$P_{k,n}$', rotation=0, ha='right')
        handles, labels = axes[3].get_legend_handles_labels()
        unique_labels = {}
        for h, l in zip(handles, labels):
            unique_labels[l] = h
        sorted_unique_labels = sorted(unique_labels.items())
        legend_handles = [h for l, h in sorted_unique_labels if l.isdigit()]
        legend_labels = [f'mode {l}' for l, h in sorted_unique_labels if l.isdigit()]
        if 'P1' in unique_labels: legend_handles.append(plt.Line2D([], [], color='gray', linestyle='--', marker='x')); legend_labels.append('$P_{1,n}$ (Core 1)')
        if 'P2' in unique_labels: legend_handles.append(plt.Line2D([], [], color='gray', linestyle='-', marker='o')); legend_labels.append('$P_{2,n}$ (Core 2)')
        if 'Prest' in unique_labels: legend_handles.append(plt.Line2D([], [], color='gray', linestyle=':', marker='^')); legend_labels.append('$P_{{\rm rest},n}$ (Rest)')
        axes[3].legend(handles=legend_handles, labels=legend_labels, title='Observable/Mode')
        axes[3].grid(True, linestyle='--', alpha=0.7)

    else:
        axes[0].text(0.5, 0.5, 'No data for modes 0 and 1 to plot.', horizontalalignment='center', verticalalignment='center', transform=axes[0].transAxes)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

else:
    print("No data collected for plotting. Please check the eigenvalue extraction step.")

## 5. Interpretation Summary

Based on the plots and the collected observables, we can interpret the behavior of the soft modes at $L=12$ as a function of core separation $d$. This section will serve as a placeholder for detailed conclusions on:

*   **Molecular texture-bound regime:** Is there evidence for negative $Q_T$, both cores being negative, and an appreciable texture-weighted bridge?
*   **Emerging isolated regime:** Do we observe a drop in the bridge, localization of modes to individual cores, or decoupling of texture signatures at larger $d$?
*   **Texture-weighted anisotropy:** How does $A_T$ behave compared to previous observations or raw anisotropy?

This analysis helps determine if the molecular identity persists or if a crossover to an isolated defect regime occurs at this system size.

In [ ]:

# Colab notebook template: L=12 texture-bound molecularity test
# ---------------------------------------------------------------
# Purpose:
#   Test whether the lowest soft modes remain
#   (i) shared across both cores,
#   (ii) anti-aligned with the texture on both cores,
#   (iii) connected by a texture-weighted bridge.
#
# Replace the TODO sections that depend on your data format / operator details.
# This notebook is designed to be easy to adapt rather than fully plug-and-play.

# =========================
# 0. Imports and settings
# =========================
import os
import math
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product
from scipy.sparse import coo_matrix, diags
from scipy.sparse.linalg import eigsh, LinearOperator
import time, gc, warnings
warnings.filterwarnings('ignore')
import primme

EPS = 1e-12

# Define variables needed for Hamiltonian construction (from cell d2c6ef5d and algebra setup)
m0 = 0.3
v = 2.0
w = 1.0

I2 = np.eye(2, dtype=complex)
sx = np.array([[0., 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
z2 = np.zeros((2, 2), dtype=complex)
alpha4 = [np.block([[z2, s], [s, z2]]) for s in (sx, sy, sz)]
beta4 = np.block([[I2, z2], [z2, -I2]])
ALPHA = [np.kron(a, I2).astype(complex) for a in alpha4]
BETA = np.kron(beta4, I2).astype(complex)
G5 = np.kron(beta4, I2).astype(complex)
MASS_B = [np.kron(beta4, t).astype(complex) for t in (sx, sy, sz)]
EYE8 = np.eye(8, dtype=complex)
R = 1.0
G5DIAG = np.diag(G5).real.copy()

# -------------------------
# User settings
# -------------------------
L = 12
d_list = [1, 2, 3, 4, 5, 6]
k_modes = 2              # start with 2; increase to 4 if affordable
core_radius = 1.5
bridge_radius = 1.0

# Pair axis convention: z-axis
PAIR_AXIS = 2  # 0=x, 1=y, 2=z

# Example core centers for a pair aligned along z around the lattice midpoint.
# Adjust if your geometry places the pair differently.
def default_core_centers(L, d):
    mid = np.array([L//2, L//2, L//2], dtype=float)
    c1 = mid.copy()
    c2 = mid.copy()
    c1[PAIR_AXIS] = (mid[PAIR_AXIS] - d/2.0) % L
    c2[PAIR_AXIS] = (mid[PAIR_AXIS] + d/2.0) % L
    return c1, c2

# Output table
rows = []

# =========================
# 1. Geometry helpers
# =========================
def all_sites(L):
    """Return array of all lattice coordinates, shape [N, 3]."""
    xs = np.arange(L)
    grid = np.stack(np.meshgrid(xs, xs, xs, indexing="ij"), axis=-1)
    return grid.reshape(-1, 3).astype(float)

SITES = all_sites(L)  # shape [Nsites, 3]
NSITES = SITES.shape[0]

def min_image_delta(coord, center, L):
    """
    Periodic minimum-image displacement from center to coord.
    coord, center: arrays [..., 3]
    returns displacement in [-L/2, L/2)
    """
    delta = coord - center
    delta = (delta + L/2.0) % L - L/2.0
    return delta

def radial_mask_about_center(sites, center, radius, L):
    """Boolean mask for sites within Euclidean radius of center (min-image)."""
    dxyz = min_image_delta(sites, center, L)
    r2 = np.sum(dxyz**2, axis=1)
    return r2 <= radius**2

def bridge_mask(sites, c1, c2, bridge_radius, L, axis=2):
    """
    Tube/corridor between c1 and c2 along chosen pair axis with min-image geometry.
    Assumes pair is mostly separated along `axis`.
    """
    # Midpoint in periodic sense:
    # Use c1 as reference, move c2 by min-image, then midpoint.
    disp = min_image_delta(c2, c1, L)
    mid = (c1 + 0.5 * disp) % L

    # Coordinates relative to midpoint
    rel = min_image_delta(sites, mid, L)

    # Longitudinal coordinate along pair axis
    z = rel[:, axis]

    # Pair half-separation along min-image direction
    halfsep = abs(disp[axis]) / 2.0

    # Perp radius
    perp_axes = [0, 1, 2]
    perp_axes.remove(axis)
    rperp2 = rel[:, perp_axes[0]]**2 + rel[:, perp_axes[1]]**2

    # Between cores longitudinally and within transverse radius
    return (np.abs(z) <= halfsep) & (rperp2 <= bridge_radius**2)

def centered_coords(sites, c1, c2, L, axis=2):
    """
    Coordinates relative to pair midpoint in min-image convention.
    Useful for second moments.
    """
    disp = min_image_delta(c2, c1, L)
    mid = (c1 + 0.5 * disp) % L
    rel = min_image_delta(sites, mid, L)
    return rel  # shape [N,3]

# =========================
# 2. Hamiltonian and texture field functions (from cell EwOP6aURTJGR and b2b4a93a)
# =========================
def hedgehog(x, y, z, cx, cy, cz, L, w):
    rx = (x - cx) - L * int(round((x - cx) / float(L)))
    ry = (y - cy) - L * int(round((y - cy) / float(L)))
    rz = (z - cz) - L * int(round((z - cz) / float(L)))
    rr = np.sqrt(rx*rx + ry*ry + rz*rz + 1e-16)
    f = np.tanh(rr / w)
    th = np.arccos(np.clip(rz / rr, -1., 1.))
    ph = np.arctan2(ry, rx)
    hx = np.sin(th)*np.cos(ph); hy = np.sin(th)*np.sin(ph); hz = np.cos(th)
    d = rr / np.sqrt(rr*rr + 0.08)
    hx, hy, hz = d*hx, d*hy, d*hz
    n = np.sqrt(hx*hx + hy*hy + hz*hz + 1e-30)
    return f, hx/n, hy/n, hz/n

def precompute_onsite(L, m0, v, w, sep):
    cx1, cy1, cz1 = L//2, L//2, (L//2 - sep//2) % L
    cx2, cy2, cz2 = L//2, L//2, (L//2 + sep//2) % L
    M = np.zeros((L, L, L, 8, 8), dtype=complex)
    base = m0*BETA + 3*R*EYE8
    for x, y, z in product(range(L), repeat=3):
        Md = base.copy()
        for sign, cx, cy, cz in [(+1, cx1, cy1, cz1), (-1, cx2, cy2, cz2)]:
            f, hx, hy, hz = hedgehog(x, y, z, cx, cy, cz, L, w)
            if f < 1e-14:
                continue
            Md = Md + sign*v*f*(hx*MASS_B[0] + hy*MASS_B[1] + hz*MASS_B[2])
        M[x, y, z] = Md
    return M, (cx1, cy1, cz1), (cx2, cy2, cz2)

def apply_H(psi_flat, L, M_on):
    psi = psi_flat.reshape((L, L, L, 8))
    out = np.einsum('xyzab,xyzb->xyza', M_on, psi)
    for mu, axis in enumerate([0, 1, 2]):
        Aplus = -0.5*(R*EYE8 - ALPHA[mu])
        Aminus = -0.5*(R*EYE8 + ALPHA[mu])
        out = out + np.einsum('ab,xyzb->xyza', Aplus, np.roll(psi, -1, axis=axis))
        out = out + np.einsum('ab,xyzb->xyza', Aminus, np.roll(psi, +1, axis=axis))
    out = np.einsum('ab,xyzb->xyza', G5, out)
    return out.reshape(-1)

def get_local_mass_texture_operator_field(L, w, current_sep):
    """Precomputes the local mass texture operator M for each site."""
    M_field = np.empty((L, L, L, 8, 8), dtype=complex)

    # Assume cx_global, cy_global are L//2
    cx_global, cy_global = L // 2, L // 2

    c1_z = (L // 2 - current_sep // 2) % L
    c2_z = (L // 2 + current_sep // 2) % L

    for x, y, z in product(range(L), repeat=3):
        vx = vy = vz = 0.0
        for sign, cx, cy, cz in [(+1, cx_global, cy_global, c1_z), (-1, cx_global, cy_global, c2_z)]:
            f, hx, hy, hz = hedgehog(x, y, z, cx, cy, cz, L, w)
            vx += sign * f * hx
            vy += sign * f * hy
            vz += sign * f * hz
        n = np.sqrt(vx * vx + vy * vy + vz * vz + 1e-30)
        if n > 1e-18: # Avoid division by zero
            M_field[x, y, z] = (vx / n) * MASS_B[0] + (vy / n) * MASS_B[1] + (vz / n) * MASS_B[2]
        else:
            M_field[x, y, z] = np.zeros((8, 8), dtype=complex)

    return M_field


# You must adapt the following loaders to your file format.

def load_mode_data(L, d, k=2):
    """
    Replaced with actual data loading and eigenvalue extraction.
    Expected return:
      eigvals: np.ndarray shape [k]
      eigvecs: np.ndarray shape [k, Nsites, Nspin]
      texture_field: object/array needed by local_texture_alignment()
    """
    # Assume m0, v, w are defined globally from earlier cells (e.g., cell `px3JZ36pTN2J`)
    # Rebuild Hamiltonian for current separation
    M_on_sep, _, _ = precompute_onsite(L, m0, v, w, d)
    N_total = 8 * L**3

    def matvec_sep(x):
        return apply_H(x, L, M_on_sep)

    Hop_sep = LinearOperator((N_total, N_total), matvec=matvec_sep, dtype=np.complex128)

    # Using parameters that showed some success, or increased maxiter
    # Note: maxiter might need tuning for larger L values and different d
    eigvals_raw, eigvecs_raw = primme.eigsh(
        Hop_sep, k=k, which='SM',
        tol=5e-4, maxiter=150000, # Increased maxiter for better convergence
        method='PRIMME_JDQMR',
    )

    # Sort eigenvalues by absolute value and ensure they are real (or close to it)
    ord_idx = np.argsort(np.abs(eigvals_raw))
    eigvals = np.real(eigvals_raw[ord_idx]) # Store real part of eigenvalues
    eigvecs_reshaped = eigvecs_raw[:, ord_idx].T.reshape(k, NSITES, 8)

    # Precompute local mass texture operator field once per separation
    texture_field = get_local_mass_texture_operator_field(L, w, d)

    return eigvals, eigvecs_reshaped, texture_field

# =========================
# 3. Local observables
# =========================
def local_density(psi):
    """
    psi shape [Nsites, Nspin] complex
    returns rho[x] = psi^\dagger psi
    """
    return np.sum(np.conjugate(psi) * psi, axis=1).real

def local_texture_alignment(psi, texture_field):
    """
    Replaced with actual texture-chirality/alignment definition \chi_T(x).
    Expected output:
      chiT: np.ndarray shape [Nsites]
    Sign convention assumed here:
      chiT < 0 means anti-aligned with the texture
    """
    # psi: shape [Nsites, Nspin]
    # texture_field: shape [L, L, L, 8, 8]

    chiT = np.zeros(NSITES, dtype=float)
    psi_reshaped = psi.reshape(L, L, L, 8) # Reshape for easier indexing with texture_field

    # Iterate through each site to compute local texture alignment
    for i, (x, y, z) in enumerate(SITES.astype(int)):
        # The per-site contribution from chi_mass_texture
        chiT[i] = np.vdot(psi_reshaped[x, y, z], texture_field[x, y, z] @ psi_reshaped[x, y, z]).real
    return chiT

# Optional: gamma5 chirality control if you want it later
def local_gamma5_chirality(psi, gamma5):
    """
    psi shape [Nsites, Nspin]
    gamma5 shape [Nspin, Nspin]
    returns chi5[x] = psi^\dagger gamma5 psi
    """
    gpsi = psi @ gamma5.T
    return np.sum(np.conjugate(psi) * gpsi, axis=1).real

# =========================
# 4. Reduction for one mode
# =========================
def reduce_mode_observables(L, d, n, eigval, psi, texture_field,
                            c1, c2, core_radius=1.5, bridge_radius=1.0, axis=2):
    rho = local_density(psi)
    # Normalize defensively in case the eigenvector is not already normalized
    nrho = rho.sum()
    if nrho > 0:
        rho = rho / nrho

    chiT = local_texture_alignment(psi, texture_field)

    # Positive anti-alignment weight
    # If chiT is not bounded/scaled the way you want, modify here.
    wT = rho * np.maximum(0.0, -chiT)

    C1 = radial_mask_about_center(SITES, c1, core_radius, L)
    C2 = radial_mask_about_center(SITES, c2, core_radius, L)
    B  = bridge_mask(SITES, c1, c2, bridge_radius, L, axis=axis)

    P1 = rho[C1].sum()
    P2 = rho[C2].sum()
    Prest = max(0.0, 1.0 - P1 - P2)

    X1 = chiT[C1].sum()
    X2 = chiT[C2].sum()
    QT = chiT.sum()

    t1 = X1 / (P1 + EPS)
    t2 = X2 / (P2 + EPS)

    # sign product
    def sgn(x, tol=1e-14):
        if x > tol: return 1
        if x < -tol: return -1
        return 0
    ST = sgn(t1) * sgn(t2)

    Brho = rho[B].sum()
    BT = wT[B].sum()

    rel = centered_coords(SITES, c1, c2, L, axis=axis)
    z = rel[:, axis]
    perp_axes = [0, 1, 2]
    perp_axes.remove(axis)
    r2_perp = rel[:, perp_axes[0]]**2 + rel[:, perp_axes[1]]**2

    # Raw anisotropy A
    A = (np.sum((z**2) * rho) /
         (np.sum(r2_perp * rho) + EPS))

    # Texture-weighted anisotropy A_T
    AT = (np.sum((z**2) * wT) /
          (np.sum(r2_perp * wT) + EPS))

    # Texture dipole along axis
    DT = np.sum(z * chiT)

    return {
        "L": L,
        "d": d,
        "n": n,
        "lambda": float(np.real(eigval)),
        "P1": float(P1),
        "P2": float(P2),
        "Prest": float(Prest),
        "QT": float(QT),
        "X1": float(X1),
        "X2": float(X2),
        "t1": float(t1),
        "t2": float(t2),
        "ST": int(ST),
        "Brho": float(Brho),
        "BT": float(BT),
        "A": float(A),
        "AT": float(AT),
        "DT": float(DT),
    }

# =========================
# 5. Main scan loop
# =========================
for d in d_list:
    c1, c2 = default_core_centers(L, d)

    # Define m0, v, w for use in load_mode_data. These should be available globally
    # from previous cells in the notebook where they are defined (e.g., px3JZ36pTN2J).
    # If not, they would need to be passed as arguments or re-defined here.
    # For consistency with the existing notebook, assuming they are global:
    # m0 = 0.3
    # v = 2.0
    # w = 1.0

    eigvals, eigvecs, texture_field = load_mode_data(L, d, k=k_modes)

    for n in range(k_modes):
        row = reduce_mode_observables(
            L=L,
            d=d,
            n=n,
            eigval=eigvals[n],
            psi=eigvecs[n], # Pass the specific eigenvector for mode n
            texture_field=texture_field,
            c1=c1,
            c2=c2,
            core_radius=core_radius,
            bridge_radius=bridge_radius,
            axis=PAIR_AXIS
        )
        rows.append(row)

# Collect results
df = pd.DataFrame(rows)
display(df.head())
print("Rows:", len(df))

# Save
os.makedirs("outputs", exist_ok=True)
csv_path = f"outputs/L{L}_texture_bridge_scan.csv"
df.to_csv(csv_path, index=False)
print("Saved:", csv_path)

# =========================
# 6. Quick summaries
# =========================
print("\nPer-d, per-mode summary:")
display(df.sort_values(["d", "n"]))

# Mean over soft pair (modes 0 and 1)
pair_df = (
    df.groupby(["L", "d"])
      .agg({
          "lambda": list,
          "P1": "mean",
          "P2": "mean",
          "Prest": "mean",
          "QT": "mean",
          "t1": "mean",
          "t2": "mean",
          "Brho": "mean",
          "BT": "mean",
          "A": "mean",
          "AT": "mean",
          "DT": "mean",
      })
      .reset_index()
)

# Add gap if k_modes >= 2
if k_modes >= 2:
    def gap_from_list(lst):
        if len(lst) < 2:
            return np.nan
        vals = sorted(lst)
        return abs(vals[1] - vals[0])
    pair_df["gap"] = pair_df["lambda"].apply(gap_from_list)

display(pair_df)

# =========================
# 7. Plots
# =========================
def plot_mode_vs_d(df, ycol, title=None):
    plt.figure(figsize=(6,4))
    for n in sorted(df["n"].unique()):
        sub = df[df["n"] == n].sort_values("d")
        plt.plot(sub["d"], sub[ycol], marker="o", label=f"mode {n}")
    plt.xlabel("d")
    plt.ylabel(ycol)
    plt.title(title or ycol)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

plot_mode_vs_d(df, "QT", title="Global texture charge QT vs d")
plot_mode_vs_d(df, "t1", title="Core-1 normalized texture alignment t1 vs d")
plot_mode_vs_d(df, "t2", title="Core-2 normalized texture alignment t2 vs d")
plot_mode_vs_d(df, "BT", title="Texture-weighted bridge BT vs d")
plot_mode_vs_d(df, "Brho", title="Plain bridge density Brho vs d")
plot_mode_vs_d(df, "AT", title="Texture-weighted anisotropy AT vs d")
plot_mode_vs_d(df, "A", title="Raw anisotropy A vs d")
plot_mode_vs_d(df, "Prest", title="Prest vs d")

if k_modes >= 2 and "gap" in pair_df.columns:
    plt.figure(figsize=(6,4))
    plt.plot(pair_df["d"], pair_df["gap"], marker="o")
    plt.xlabel("d")
    plt.ylabel("gap = |lambda1-lambda0|")
    plt.title("Soft-pair gap vs d")
    plt.grid(True, alpha=0.3)
    plt.show()

# =========================
# 8. Interpretation helpers
# =========================
def classify_row(row):
    """
    Very simple rule-based label for quick triage.
    Adjust thresholds after first data look.
    """
    both_anti = (row["t1"] < 0) and (row["t2"] < 0) and (row["ST"] == 1)
    shared = (row["P1"] > 0.05) and (row["P2"] > 0.05)
    bridged = (row["BT"] > 1e-4)  # tune threshold after seeing magnitudes

    if both_anti and shared and bridged:
        return "texture-bound molecular"
    elif both_anti and shared:
        return "shared anti-aligned"
    elif (row["P1"] > 0.1 and row["P2"] < 0.03) or (row["P2"] > 0.1 and row["P1"] < 0.03):
        return "possibly localized"
    else:
        return "mixed/inconclusive"

df["class"] = df.apply(classify_row, axis=1)
display(df[["d", "n", "lambda", "P1", "P2", "Prest", "QT", "t1", "t2", "ST", "BT", "class"]])

# =========================
# 9. Optional correlation checks
# =========================
def corr_safe(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 3 or np.std(x) < 1e-15 or np.std(y) < 1e-15:
        return np.nan
    return np.corrcoef(x, y)[0,1]

print("Correlation checks:")
print("A vs AT:", corr_safe(df["A"], df["AT"]))
print("lambda vs QT:", corr_safe(df["lambda"], df["QT"]))
print("lambda vs BT:", corr_safe(df["lambda"], df["BT"]))
print("Prest vs BT:", corr_safe(df["Prest"], df["BT"]))
if "gap" in pair_df.columns:
    print("gap vs mean BT:", corr_safe(pair_df["gap"], pair_df["BT"]))
    print("gap vs mean QT:", corr_safe(pair_df["gap"], pair_df["QT"]))
    print("gap vs mean AT:", corr_safe(pair_df["gap"], pair_df["AT"]))

# =========================
# 10. Representative site maps (optional)
# =========================
# If you want 2D slices through the midpoint plane or through the pair axis plane,
# adapt these plotting routines to your preferred orientation.

def reshape_field(field, L):
    return field.reshape(L, L, L)

def plot_midplane(field, L, axis=2, index=None, title="field"):
    arr = reshape_field(field, L)
    if index is None:
        index = L // 2
    if axis == 0:
        sl = arr[index, :, :]
    elif axis == 1:
        sl = arr[:, index, :]
    else:
        sl = arr[:, :, index]

    plt.figure(figsize=(5,4))
    plt.imshow(sl.T, origin="lower", cmap="coolwarm")
    plt.colorbar()
    plt.title(f"{title}, axis={axis}, slice={index}")
    plt.show()

# Example usage after you load one mode manually:
# d = 3
# c1, c2 = default_core_centers(L, d)
# eigvals, eigvecs, texture_field = load_mode_data(L, d, k=k_modes)
# psi = eigvecs[0]
# rho = local_density(psi); rho /= rho.sum()
# chiT = local_texture_alignment(psi, texture_field)
# plot_midplane(rho, L, axis=1, title=f"rho mode0 d={d}")
# plot_midplane(chiT, L, axis=1, title=f"chiT mode0 d={d}")

print("\nNotebook template ready.")
print("Next: implement load_mode_data() and local_texture_alignment().")